In [1]:
import os
import math
import random
import re
import string
import hashlib
from collections import deque, defaultdict
from dataclasses import dataclass
from typing import Callable, Dict, List, Optional, Tuple
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.nn import functional as F
from model import GPTModel
from tokenizer import CharTokenizer
from dataclass import Instance, Task
from registry import TASKS
from save_data import save_mixed_trace_file

task_name = "count_char"

In [12]:
DATASET_SIZE = 50000
CORRECT_RATIO = [0.0,  0.1, 0.2, 0.3, 0.4, 0.5, 0.55, 0.6, 0.7, 0.8, 0.9, 1.0]

task_name = "count_char"
instances = [TASKS[task_name].sample() for _ in range(DATASET_SIZE)]

for rho in CORRECT_RATIO:
    save_mixed_trace_file(
        instances,
        correct_ratio=rho,
        output_file=f"{task_name}/{task_name}_rho_{int(rho*100)}.txt",
        seed=100,
    )

Saved: count_char/count_char_rho_0.txt
Total: 50000
Correct: 0 (0.00%)
Wrong: 50000 (100.00%)
Saved: count_char/count_char_rho_10.txt
Total: 50000
Correct: 5000 (10.00%)
Wrong: 45000 (90.00%)
Saved: count_char/count_char_rho_20.txt
Total: 50000
Correct: 10000 (20.00%)
Wrong: 40000 (80.00%)
Saved: count_char/count_char_rho_30.txt
Total: 50000
Correct: 15000 (30.00%)
Wrong: 35000 (70.00%)
Saved: count_char/count_char_rho_40.txt
Total: 50000
Correct: 20000 (40.00%)
Wrong: 30000 (60.00%)
Saved: count_char/count_char_rho_50.txt
Total: 50000
Correct: 25000 (50.00%)
Wrong: 25000 (50.00%)
Saved: count_char/count_char_rho_55.txt
Total: 50000
Correct: 27500 (55.00%)
Wrong: 22500 (45.00%)
Saved: count_char/count_char_rho_60.txt
Total: 50000
Correct: 30000 (60.00%)
Wrong: 20000 (40.00%)
Saved: count_char/count_char_rho_70.txt
Total: 50000
Correct: 35000 (70.00%)
Wrong: 15000 (30.00%)
Saved: count_char/count_char_rho_80.txt
Total: 50000
Correct: 40000 (80.00%)
Wrong: 10000 (20.00%)
Saved: count_cha

In [9]:

from tqdm.auto import tqdm
task_name = "count_char"
rho = 1.0
data_file = f"{task_name}/{task_name}_rho_{int(rho*100)}.txt"

batch_size = 256
steps = 6000
learning_rate = 3e-4
min_learning_rate = 1e-5
warmup_steps = 600
weight_decay = 0.01
max_grad_norm = 1.0

n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.05

model_seed = 2200
val_seed = 99999
val_size = 1000
val_batch_size = 256

USE_COMPILE = True

device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    torch.set_float32_matmul_precision("high")

use_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()

task = TASKS[task_name]
tokenizer = task.tokenizer

block_size = task.block_size
max_new_tokens = task.max_new_tokens
pad_id = tokenizer.pad_id
newline_id = tokenizer.newline_id
vocab_size = tokenizer.vocab_size

print("Device:", device)
print("BF16:", use_bf16)


with open(data_file, "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

print("Training file:", data_file)
print("Training examples:", len(lines))


train_prompts = {line.split(" ", 1)[0] for line in lines}
train_words = {prompt.split(";")[0] for prompt in train_prompts}


random.seed(val_seed)

val_instances = []
val_words = set()

while len(val_instances) < val_size:
    inst = task.sample()
    word = inst.prompt.split(";")[0]

    if word in train_words or word in val_words:
        continue

    val_words.add(word)
    val_instances.append(inst)

print("Validation examples:", len(val_instances))
print("Train/validation word overlap:", len(train_words & val_words))

assert train_words.isdisjoint(val_words)


encoded = []
max_seq_len = 0

for line in tqdm(lines, desc="Encoding"):
    prompt, continuation = line.split(" ", 1)

    p_ids = tokenizer.encode(prompt)
    t_ids = tokenizer.encode(" " + continuation + "\n")

    full = p_ids + t_ids

    if len(full) > block_size:
        raise ValueError(f"Sequence length {len(full)} exceeds block size {block_size}")

    x = full[:-1]
    y = full[1:]
    mask = [1.0 if i + 1 >= len(p_ids) else 0.0 for i in range(len(x))]

    encoded.append((x, y, mask))
    max_seq_len = max(max_seq_len, len(x))


print("Block size:", block_size)
print("Actual sequence length:", max_seq_len)


xs, ys, masks = [], [], []

for x, y, mask in tqdm(encoded, desc="Tensorizing"):
    pad = max_seq_len - len(x)

    xs.append(x + [pad_id] * pad)
    ys.append(y + [pad_id] * pad)
    masks.append(mask + [0.0] * pad)


xs = torch.tensor(xs, dtype=torch.long, device=device)
ys = torch.tensor(ys, dtype=torch.long, device=device)
masks = torch.tensor(masks, dtype=torch.float32, device=device)

del encoded, lines

print("Train tensor:", xs.shape)


random.seed(model_seed)
torch.manual_seed(model_seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(model_seed)


base_model = GPTModel(
    vocab_size=vocab_size,
    block_size=block_size,
    pad_id=pad_id,
    n_embd=n_embd,
    n_head=n_head,
    n_layer=n_layer,
    dropout=dropout
).to(device)

print("Parameters:", sum(p.numel() for p in base_model.parameters() if p.requires_grad))


model = torch.compile(base_model) if USE_COMPILE and device == "cuda" else base_model


try:
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
        fused=(device == "cuda")
    )
except:
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )


model.train()

pbar = tqdm(range(steps), desc=f"Training rho={rho}")

for step in pbar:

    if step < warmup_steps:
        lr = learning_rate * (step + 1) / warmup_steps
    else:
        decay = (step - warmup_steps) / (steps - warmup_steps)
        coeff = 0.5 * (1.0 + math.cos(math.pi * decay))
        lr = min_learning_rate + coeff * (learning_rate - min_learning_rate)

    for group in optimizer.param_groups:
        group["lr"] = lr

    ix = torch.randint(0, xs.shape[0], (batch_size,), device=device)

    xb = xs[ix]
    yb = ys[ix]
    mb = masks[ix]

    optimizer.zero_grad(set_to_none=True)

    if use_bf16:
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            _, loss = model(xb, targets=yb, mask=mb)
    else:
        _, loss = model(xb, targets=yb, mask=mb)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
    optimizer.step()

    if step % 50 == 0:
        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{lr:.2e}")


print("\nTraining finished.")
print("Running validation once...")


base_model.eval()

correct = 0
separator_count = 0
from collections import defaultdict

base_model.eval()

correct = 0
separator_count = 0

val_groups = defaultdict(list)

for inst in val_instances:
    ids = tokenizer.encode(inst.prompt)
    val_groups[len(ids)].append((inst, ids))

print("Validation prompt lengths:", sorted(val_groups.keys()))

with torch.inference_mode():

    for prompt_len, group in tqdm(val_groups.items(), desc="Validation"):

        for start in range(0, len(group), val_batch_size):

            batch = group[start:start + val_batch_size]

            batch_instances = [x[0] for x in batch]
            batch_ids = [x[1] for x in batch]

            context = torch.tensor(
                batch_ids,
                dtype=torch.long,
                device=device
            )

            out = base_model.generate(
                context,
                max_new_tokens=max_new_tokens,
                stop_id=newline_id,
                greedy=True
            )

            for row, inst in zip(out.tolist(), batch_instances):

                text = tokenizer.decode(row)
                generated = text[len(inst.prompt):].split("\n", 1)[0]

                if ":" in generated:
                    separator_count += 1
                    answer = generated.rsplit(":", 1)[1]
                    nums = re.findall(r"\d+", answer)
                    pred = nums[0] if nums else None
                else:
                    pred = None

                correct += int(pred == inst.gold)


val_acc = correct / len(val_instances)
separator_rate = separator_count / len(val_instances)

print()
print(f"ρ = {rho:.1f}")
print(f"Validation accuracy = {val_acc*100:.2f}%")
print(f"Separator rate = {separator_rate*100:.2f}%")

Device: cuda
BF16: True
Training file: count_char/count_char_rho_100.txt
Training examples: 80000
Validation examples: 1000
Train/validation word overlap: 0


Encoding:   0%|          | 0/80000 [00:00<?, ?it/s]

Block size: 76
Actual sequence length: 48


Tensorizing:   0%|          | 0/80000 [00:00<?, ?it/s]

Train tensor: torch.Size([80000, 48])
Parameters: 813358


Training rho=1.0:   0%|          | 0/6000 [00:00<?, ?it/s]


Training finished.
Running validation once...
Validation prompt lengths: [8, 9, 10, 11, 12, 13, 14, 15, 16]


Validation:   0%|          | 0/9 [00:00<?, ?it/s]


ρ = 1.0
Validation accuracy = 68.00%
Separator rate = 100.00%


In [11]:
import torch

task_name = "count_char"

task = TASKS[task_name]
tokenizer = task.tokenizer

newline_id = tokenizer.encode("\n")[0]

test_prompt = "oqrrgkwxgzm;k"

context = torch.tensor(
    [tokenizer.encode(test_prompt)],
    dtype=torch.long,
    device=device
)

base_model.eval()

with torch.no_grad():
    generated = base_model.generate(
        context,
        max_new_tokens=50,
        stop_id=newline_id,
        greedy=True
    )[0].tolist()

text = tokenizer.decode(generated)

print("Model output:")
print(text)

word, char = test_prompt.split(";")
gold = word.count(char)

print("Gold answer:", gold)

Model output:
oqrrgkwxgzm;k o q r r g k w x g z m : 1

Gold answer: 1


In [2]:
import math, random, re, gc, torch
import matplotlib.pyplot as plt

from collections import defaultdict
from tqdm.auto import tqdm
from model import GPTModel
from registry import TASKS


task_name = "count_char"

rho_values = [
    0.0, 0.1, 0.2, 0.3, 0.4, 0.45,
    0.5, 0.6, 0.7, 0.8, 0.9, 1.0
]

batch_size = 128
steps = 6000

learning_rate = 3e-4
min_learning_rate = 1e-5
warmup_steps = 600
weight_decay = 0.01
max_grad_norm = 1.0

n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.05

model_seed = 2100
val_seed = 99999

val_size = 1000
val_batch_size = 256

USE_COMPILE = True

device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    torch.set_float32_matmul_precision("high")

use_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()

task = TASKS[task_name]
tokenizer = task.tokenizer

block_size = task.block_size
max_new_tokens = task.max_new_tokens

pad_id = tokenizer.pad_id
newline_id = tokenizer.newline_id
vocab_size = tokenizer.vocab_size

print("Device:", device)
print("BF16:", use_bf16)


first_file = f"{task_name}/{task_name}_rho_0.txt"

with open(first_file, "r", encoding="utf-8") as f:
    first_lines = [line.strip() for line in f if line.strip()]

train_prompts = {line.split(" ", 1)[0] for line in first_lines}
train_words = {prompt.split(";")[0] for prompt in train_prompts}


random.seed(val_seed)

val_instances = []
val_words = set()

while len(val_instances) < val_size:

    inst = task.sample()
    word = inst.prompt.split(";")[0]

    if word in train_words or word in val_words:
        continue

    val_words.add(word)
    val_instances.append(inst)


print("Validation examples:", len(val_instances))
print("Train/validation overlap:", len(train_words & val_words))

assert train_words.isdisjoint(val_words)


val_groups = defaultdict(list)

for inst in val_instances:
    ids = tokenizer.encode(inst.prompt)
    val_groups[len(ids)].append((inst, ids))

print("Validation prompt lengths:", sorted(val_groups.keys()))


accuracies = []
separator_rates = []


for rho in rho_values:

    print("\n" + "=" * 70)
    print(f"ρ = {rho:.2f}")
    print("=" * 70)

    rho_id = int(round(rho * 100))

    data_file = f"{task_name}/{task_name}_rho_{rho_id}.txt"

    with open(data_file, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    print("Training file:", data_file)
    print("Examples:", len(lines))


    encoded = []
    max_seq_len = 0

    for line in tqdm(
        lines,
        desc=f"Encoding ρ={rho:.2f}",
        leave=False
    ):

        prompt, continuation = line.split(" ", 1)

        p_ids = tokenizer.encode(prompt)
        t_ids = tokenizer.encode(" " + continuation + "\n")

        full = p_ids + t_ids

        if len(full) > block_size:
            raise ValueError(
                f"Sequence length {len(full)} > "
                f"block size {block_size}"
            )

        x = full[:-1]
        y = full[1:]

        mask = [
            1.0 if i + 1 >= len(p_ids) else 0.0
            for i in range(len(x))
        ]

        encoded.append((x, y, mask))

        max_seq_len = max(
            max_seq_len,
            len(x)
        )


    xs, ys, masks = [], [], []

    for x, y, mask in encoded:

        pad = max_seq_len - len(x)

        xs.append(
            x + [pad_id] * pad
        )

        ys.append(
            y + [pad_id] * pad
        )

        masks.append(
            mask + [0.0] * pad
        )


    xs = torch.tensor(
        xs,
        dtype=torch.long,
        device=device
    )

    ys = torch.tensor(
        ys,
        dtype=torch.long,
        device=device
    )

    masks = torch.tensor(
        masks,
        dtype=torch.float32,
        device=device
    )

    del encoded, lines

    print("Train tensor:", xs.shape)


    random.seed(model_seed)
    torch.manual_seed(model_seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(model_seed)


    base_model = GPTModel(
        vocab_size=vocab_size,
        block_size=block_size,
        pad_id=pad_id,
        n_embd=n_embd,
        n_head=n_head,
        n_layer=n_layer,
        dropout=dropout
    ).to(device)


    model = (
        torch.compile(base_model)
        if USE_COMPILE and device == "cuda"
        else base_model
    )


    try:

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay,
            fused=(device == "cuda")
        )

    except:

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay
        )


    model.train()

    pbar = tqdm(
        range(steps),
        desc=f"Training ρ={rho:.2f}"
    )


    for step in pbar:

        if step < warmup_steps:

            lr = (
                learning_rate
                * (step + 1)
                / warmup_steps
            )

        else:

            decay = (
                (step - warmup_steps)
                / (steps - warmup_steps)
            )

            coeff = 0.5 * (
                1.0
                + math.cos(
                    math.pi * decay
                )
            )

            lr = (
                min_learning_rate
                + coeff
                * (
                    learning_rate
                    - min_learning_rate
                )
            )


        for group in optimizer.param_groups:
            group["lr"] = lr


        ix = torch.randint(
            0,
            xs.shape[0],
            (batch_size,),
            device=device
        )

        xb = xs[ix]
        yb = ys[ix]
        mb = masks[ix]


        optimizer.zero_grad(
            set_to_none=True
        )


        if use_bf16:

            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16
            ):

                _, loss = model(
                    xb,
                    targets=yb,
                    mask=mb
                )

        else:

            _, loss = model(
                xb,
                targets=yb,
                mask=mb
            )


        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_grad_norm
        )

        optimizer.step()


        if step % 50 == 0:

            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                lr=f"{lr:.2e}"
            )


    print("Training finished.")
    print("Running validation...")


    base_model.eval()

    correct = 0
    separator_count = 0


    with torch.inference_mode():

        for prompt_len, group in tqdm(
            val_groups.items(),
            desc=f"Validation ρ={rho:.2f}",
            leave=False
        ):

            for start in range(
                0,
                len(group),
                val_batch_size
            ):

                batch = group[
                    start:start + val_batch_size
                ]

                batch_instances = [
                    item[0]
                    for item in batch
                ]

                batch_ids = [
                    item[1]
                    for item in batch
                ]


                context = torch.tensor(
                    batch_ids,
                    dtype=torch.long,
                    device=device
                )


                out = base_model.generate(
                    context,
                    max_new_tokens=max_new_tokens,
                    stop_id=newline_id,
                    greedy=True
                )


                for row, inst in zip(
                    out.tolist(),
                    batch_instances
                ):

                    text = tokenizer.decode(row)

                    generated = (
                        text[len(inst.prompt):]
                        .split("\n", 1)[0]
                    )


                    if ":" in generated:

                        separator_count += 1

                        answer = (
                            generated
                            .rsplit(":", 1)[1]
                        )

                        nums = re.findall(
                            r"\d+",
                            answer
                        )

                        pred = (
                            nums[0]
                            if nums
                            else None
                        )

                    else:

                        pred = None


                    correct += int(
                        pred == inst.gold
                    )


    val_acc = (
        correct
        / len(val_instances)
    )

    separator_rate = (
        separator_count
        / len(val_instances)
    )


    accuracies.append(val_acc)
    separator_rates.append(separator_rate)


    print(
        f"ρ={rho:.2f} | "
        f"accuracy={val_acc*100:.2f}% | "
        f"separator={separator_rate*100:.2f}%"
    )


    del model
    del base_model
    del optimizer
    del xs
    del ys
    del masks

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("\n" + "=" * 70)
print("FINAL RESULTS")
print("=" * 70)

for rho, acc, sep in zip(
    rho_values,
    accuracies,
    separator_rates
):

    print(
        f"ρ={rho:.2f} | "
        f"accuracy={acc*100:.2f}% | "
        f"separator={sep*100:.2f}%"
    )



Device: cuda
BF16: True
Validation examples: 1000
Train/validation overlap: 0
Validation prompt lengths: [8, 9, 10, 11, 12, 13, 14, 15, 16]

ρ = 0.00
Training file: count_char/count_char_rho_0.txt
Examples: 50000


Encoding ρ=0.00:   0%|          | 0/50000 [00:00<?, ?it/s]

Train tensor: torch.Size([50000, 48])


Training ρ=0.00:   0%|          | 0/6000 [00:00<?, ?it/s]

Training finished.
Running validation...


...

Training ρ=0.40:   0%|          | 0/6000 [00:00<?, ?it/s]

Training finished.
Running validation...


Validation ρ=0.40:   0%|          | 0/9 [00:00<?, ?it/s]

ρ=0.40 | accuracy=68.10% | separator=100.00%

ρ = 0.45


FileNotFoundError: [Errno 2] No such file or directory: 'count_char/count_char_rho_45.txt'

In [ ]:

plt.figure(figsize=(8, 5))

plt.plot(
    rho_values,
    [100 * acc for acc in accuracies],
    marker="o",
    linewidth=2
)

plt.xlabel(r"Correct Trace Ratio $\rho$")
plt.ylabel("Validation Accuracy (%)")
plt.title(r"Count Character: Accuracy vs. $\rho$")

plt.xticks(rho_values)
plt.ylim(0, 100)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

NameError: name 'accuracies' is not defined

<Figure size 700x500 with 0 Axes>